In [ ]:
from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)

In [ ]:
# Tipos de tren que queremos
train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Elimination": "SUPRESIÓN",
    "End": "FIN",
    "Entry": "ENTRY",
    "Exit": "EXIT",
    "Maneuver": "MANIOBRA",
    "Platform": "ALTA",
    "PlatformForecast": "PREVISIÓN",  # "PREDICCIÓN",
    "Stopped": "STOP",
    "TrackingLost": "LOST_TRACK",
}

# Orden lógico de movimientos
mov_sorter = {
    v: k
    for k, v in enumerate(
        [
            "PREVISIÓN",
            "APROXIMACIÓN",
            "MANIOBRALLEGADA",
            "EXIT",
            "LLEGADA",
            "FIN",
            "BAJA",
            "ALTA",
            "SALIDA",
            "MANIOBRASALIDA",
            "MANIOBRA",
        ]
    )
}

In [ ]:
def cargarHistorico(
    start_date: str,
    end_date: str,
    estaciones: list[str],
    trenes: list[str],
    xSIV: bool,
    JCTC: bool,
    pro: bool = True
):
    # Comprobamos que la fecha de fin sea después de la de inicio
    if end_date <= start_date:
        end_date = (pd.to_datetime(start_date) + timedelta(days=1)).strftime(
            "%Y-%m-%d %H:%M:%S"
        )

    historico = getHistoricoMOW(
        estaciones=estaciones, trenes=trenes, inicio=start_date, fin=end_date, pro=pro,xSIV=xSIV, jCTC=JCTC
    )
    historico = historico[
        (historico["Fecha"] >= pd.to_datetime(start_date))
        & (historico["Fecha"] <= pd.to_datetime(end_date))
    ]
    # Usamos movimientos auditados
    #historico = historico[
         #np.invert(historico["FuenteVía"].isin(["PLANNED", "SITRA_PROVIDED"]))
     #]
    historico = historico[historico["NTécnico"].apply(isValidCode)].dropna(
        subset=["Movimiento"]
    )
    historico["mov_ord"] = historico["Movimiento"].apply(mov_sorter.get)
    return historico

In [ ]:
ntrenes = [rellenarId(f"{i}") for i in np.arange(100000)]
start_date = "2025-05-10"
end_date = "2025-05-11"

In [ ]:
# historico_pro = cargarHistorico(start_date, end_date, [], ntrenes, pro=True, xSIV=True, JCTC=False)
historico_pro = cargarHistorico(start_date, end_date, [], ntrenes, pro=False, xSIV=True, JCTC= False)


In [ ]:
historico_pre[historico_pre["Movimiento"] == "CIRCULATION_ORIGIN_CHANGED"]

In [ ]:
historico_pre

### Cambios circulación

In [ ]:
def procesarCambio(movimiento, historico):
    tipo_cambio = {
        "CIRCULATION_ORIGIN_CHANGED": {
            "startLocationSequence": "Secuencia",
            "departurePlannedStartLocation": "NuevaPlanificaciónInicio",
        },
        "CIRCULATION_DESTINATION_CHANGED": {
            "endLocationSequence": "Secuencia",
            "arrivalPlannedEndLocation": "NuevaPlanificaciónFin",
        },
    }

    # Creamos un dataframe a partir de la descripción de los cambios
    cambio = historico.loc[
        historico["Movimiento"].isin([movimiento])
        & historico["Descripción"].apply(bool)
    ]
    cambio = (
        cambio[["Fecha", "Movimiento", "Descripción"]]
        .apply(
            lambda x: {
                "FechaPrevisión": x["Fecha"],
                "MovimientoPrevisión": x["Movimiento"],
                **x["Descripción"],
            },
            axis=1,
        )
        .tolist()
    )
    # cambio = (
    #     pd.DataFrame(cambio)
    #     .drop(["reason"], axis=1)
    #     .rename(columns={"train": "NTécnico", **tipo_cambio[movimiento]})
    #     .drop_duplicates(subset=["NTécnico", "Secuencia"])
    # )
    # # Conversión de tipos
    # cambio["Secuencia"] = cambio["Secuencia"].astype("Int64")
    # cols = [
    #     c
    #     for c in ["NuevaPlanificaciónInicio", "NuevaPlanificaciónFin"]
    #     if c in cambio.columns
    # ]
    # cambio[cols] = cambio[cols].map(time2localtime, unit="ms")
    # return cambio

##### Origen
Mensajes donde hay "CIRCULATION_ORIGIN_CHANGED" generados por Sitra como previsión de cambio de origen. Hasta donde sé debería empezar en un punto más avanzado respecto a la planificado completo, no extenderla (ej: secuencia de inicio en 7 en vez de 1)

In [ ]:
cambio_origen = procesarCambio("CIRCULATION_ORIGIN_CHANGED", historico_pre)

In [ ]:
cambio_origen

In [ ]:
cambio_origen = (
    pd.merge(
        historico_pre[
            (historico_pre["Movimiento"].isin(["ALTA", "SALIDA"]))
        ].reset_index(),
        cambio_origen,
        how="inner",
        on=["NTécnico", "Secuencia"],
    )
    .drop(["Descripción"], axis=1)
    .drop_duplicates()
)

##### Destino
Mensajes donde hay "CIRCULATION_DESTINATION_CHANGED" generados por Sitra como previsión de cambio de destino. Hasta donde sé debería terminar antes de hacer el recorrido planificado completo, no extenderlo (ej: secuencia de fin en 76 en vez de 83)

In [ ]:
cambio_destino = procesarCambio("CIRCULATION_DESTINATION_CHANGED", historico_pre)

cambio_destino = (
    pd.merge(
        historico_pre[
            (historico_pre["Movimiento"].isin(["LLEGADA", "SUPRESIÓN", "BAJA", "FIN"]))
        ].reset_index(),
        cambio_destino,
        how="inner",
        on=["NTécnico", "Secuencia"],
    )
    .drop(["Descripción"], axis=1)
    .drop_duplicates()
)

#### Histórico actualizado

In [ ]:
historico_extra = (
    pd.concat(
        [
            historico_pre.loc[
                historico_pre.index.drop(
                    cambio_origen["index"].tolist() + cambio_destino["index"].tolist()
                )
            ],
            cambio_origen,
            cambio_destino,
        ]
    )
    .reset_index(drop=True)
    .drop(["Descripción"], axis=1)
)

cambios = historico_extra.sort_values(by=["NTécnico", "Fecha"]).dropna(
    subset=["FechaPrevisión"]
)
cambios["AnticipaciónCambio"] = (cambios["Fecha"] - cambios["FechaPrevisión"]).apply(
    lambda x: formatTimedelta(x.total_seconds())
)


In [ ]:
cambios.head(5)

In [ ]:
cambios = cambios[
    [
        "Fecha",
        "FechaPrevisión",
        "AnticipaciónCambio",
        "NTécnico",
        "Nombre",
        "Código",
        "Secuencia",
        "Movimiento",
        "FuenteMovimiento",
        "Vía",
        "FuenteVía",
        "Retraso (segundos)",
        "CategoríaCirculación",
        "Producto",
        "Empresa",
        "SalidaPlanificada",
        "MovimientoPrevisión",
        "NuevaPlanificaciónInicio",
        "NuevaPlanificaciónFin",
    ]
].drop_duplicates()

In [ ]:
guardarExcel(cambios, "CambiosCirculación.xlsx", append_sheet=False)

In [ ]:
fig = px.scatter(
    cambios,
    "Fecha",
    "FechaPrevisión",
    color="CategoríaCirculación",
)
mn = cambios[["Fecha", "FechaPrevisión"]].min(axis=None)
mx = cambios[["Fecha", "FechaPrevisión"]].max(axis=None)
fig.add_trace(
    go.Scatter(
        x=pd.to_datetime([mn, mx]),
        y=pd.to_datetime([mn, mx]),
        mode="lines",
        hoverinfo="skip",
        showlegend=False,
    )
)
annotation_text = "Si está por encima de la línea, el cambio es tarde"
fig.add_annotation(
    xref="paper",
    yref="paper",
    x=0,
    y=1.05,
    text=annotation_text,
    showarrow=False,
)
fig.show()